# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [1]:
!pip install boto3

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
from datetime import datetime
import os
import boto3
from botocore.client import Config
from torch.utils.tensorboard import SummaryWriter
import yaml
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
Using device: cuda


## 2. Подготовка данных

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        if 'train' in csv_file:
            self.subfolder = 'train_data'
        else:
            self.subfolder = 'test_data'

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_name = self.data.iloc[idx]['file_name']
        if '/' in file_name:
            file_name = file_name.split('/')[-1]
        img_path = self.root_dir / self.subfolder / file_name
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

train_dataset_2 = ImageDataset(
    csv_file='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Train_2/train.csv',
    root_dir='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Train_2',
    transform=train_transform
)

test_dataset_2 = ImageDataset(
    csv_file='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Test_2/test.csv',
    root_dir='/content/drive/MyDrive/ai-vs-human-generated-dataset-hw/Test_2',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
train_loader_2 = DataLoader(train_dataset_2, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader_2 = DataLoader(test_dataset_2, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')
print(f'Train_2 dataset size: {len(train_dataset_2)}')
print(f'Test_2 dataset size: {len(test_dataset_2)}')

Train dataset size: 9993
Test dataset size: 3997
Train_2 dataset size: 3997
Test_2 dataset size: 2000


## 3. Создание модели ResNet18

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

## 4. Определение функции потерь и оптимизатора

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
Path('logs').mkdir(exist_ok=True)
Path('saved_models').mkdir(exist_ok=True)

training_params = {
    'batch_size': batch_size,
    'num_epochs': 10,
    'learning_rate': 0.001,
    'device': str(device),
    'timestamp': datetime.now().isoformat()
}

with open('logs/training_params.json', 'w') as f:
    json.dump(training_params, f, indent=2)

## 5. Функция обучения

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted')
    epoch_recall = recall_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [ ]:
num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []
val_losses = []
val_accs = []
val_f1s = []

writer = SummaryWriter('logs/tensorboard')

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)

    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_precision, val_recall = validate(model, test_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)

    writer.add_scalar('Train/Loss', train_loss, epoch)
    writer.add_scalar('Train/Accuracy', train_acc, epoch)
    writer.add_scalar('Train/F1', train_f1, epoch)
    writer.add_scalar('Validation/Loss', val_loss, epoch)
    writer.add_scalar('Validation/Accuracy', val_acc, epoch)
    writer.add_scalar('Validation/F1', val_f1, epoch)
    writer.add_scalar('Validation/Precision', val_precision, epoch)
    writer.add_scalar('Validation/Recall', val_recall, epoch)

    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}')

print('\nTraining completed!')


Epoch 1/10
--------------------------------------------------


Validation: 100%|██████████| 125/125 [05:42<00:00,  2.74s/it]


Train Loss: 0.3562, Acc: 0.8502, F1: 0.8502
Val Loss: 0.2848, Acc: 0.8864, F1: 0.8863, Precision: 0.8886, Recall: 0.8864

Epoch 2/10
--------------------------------------------------


Validation: 100%|██████████| 125/125 [00:35<00:00,  3.52it/s]


Train Loss: 0.2854, Acc: 0.8864, F1: 0.8864
Val Loss: 0.3129, Acc: 0.8872, F1: 0.8871, Precision: 0.8886, Recall: 0.8872

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0977, Acc: 0.9628, F1: 0.9628
Val Loss: 0.1167, Acc: 0.9557, F1: 0.9557, Precision: 0.9568, Recall: 0.9557

Epoch 9/10
--------------------------------------------------


Validation: 100%|██████████| 125/125 [00:33<00:00,  3.72it/s]


Train Loss: 0.0848, Acc: 0.9671, F1: 0.9671
Val Loss: 0.0915, Acc: 0.9667, F1: 0.9667, Precision: 0.9668, Recall: 0.9667

Epoch 10/10
--------------------------------------------------


Validation: 100%|██████████| 125/125 [00:32<00:00,  3.83it/s]

Train Loss: 0.0783, Acc: 0.9686, F1: 0.9686
Val Loss: 0.0899, Acc: 0.9700, F1: 0.9700, Precision: 0.9700, Recall: 0.9700

Training completed!


In [ ]:
torch.save(model.state_dict(), 'saved_models/model_resnet18_base.pth')
writer.close()

## 8. Оценка модели на тестовом датасете

In [ ]:
def evaluate_model(model, dataloader, device, dataset_name="Test"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f'{dataset_name} Evaluation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    metrics = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

    return metrics

test_metrics = evaluate_model(model, test_loader, device, "Test_1")

print(f"\nРезультаты на Test_1:")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"F1 Score: {test_metrics['f1_score']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall: {test_metrics['recall']:.4f}")

with open('logs/test1_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)

writer_test = SummaryWriter('logs/tensorboard_test')
writer_test.add_scalar('Test/Accuracy', test_metrics['accuracy'], 0)
writer_test.add_scalar('Test/F1_Score', test_metrics['f1_score'], 0)
writer_test.add_scalar('Test/Precision', test_metrics['precision'], 0)
writer_test.add_scalar('Test/Recall', test_metrics['recall'], 0)
writer_test.close()

Test_1 Evaluation: 100%|██████████| 125/125 [00:34<00:00,  3.65it/s]


Результаты на Test_1:
Accuracy: 0.9700
F1 Score: 0.9700
Precision: 0.9700
Recall: 0.9700


## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

In [ ]:
finetune_params = {
    'num_epochs': 5,
    'learning_rate': 0.0001,
    'batch_size': batch_size,
    'timestamp': datetime.now().isoformat()
}

with open('logs/finetune_params.json', 'w') as f:
    json.dump(finetune_params, f, indent=2)

model_finetuned = models.resnet18(pretrained=True)
num_features = model_finetuned.fc.in_features
model_finetuned.fc = nn.Linear(num_features, 2)
model_finetuned = model_finetuned.to(device)

model_finetuned.load_state_dict(torch.load('saved_models/model_resnet18_base.pth', map_location=device))

<All keys matched successfully>

In [ ]:
criterion_finetune = nn.CrossEntropyLoss()
optimizer_finetune = optim.Adam(model_finetuned.parameters(), lr=0.0001)
scheduler_finetune = optim.lr_scheduler.StepLR(optimizer_finetune, step_size=3, gamma=0.5)

In [ ]:
writer_finetune = SummaryWriter('logs/tensorboard_finetune')

num_epochs_finetune = 5
finetune_train_losses = []
finetune_train_accs = []
finetune_train_f1s = []
finetune_val_losses = []
finetune_val_accs = []
finetune_val_f1s = []

for epoch in range(num_epochs_finetune):
    print(f'\nFine-tuning Epoch {epoch+1}/{num_epochs_finetune}')
    print('-' * 50)

    train_loss, train_acc, train_f1 = train_epoch(model_finetuned, train_loader_2, criterion_finetune, optimizer_finetune, device)
    val_loss, val_acc, val_f1, val_precision, val_recall = validate(model_finetuned, test_loader_2, criterion_finetune, device)
    scheduler_finetune.step()

    finetune_train_losses.append(train_loss)
    finetune_train_accs.append(train_acc)
    finetune_train_f1s.append(train_f1)
    finetune_val_losses.append(val_loss)
    finetune_val_accs.append(val_acc)
    finetune_val_f1s.append(val_f1)

    writer_finetune.add_scalar('Train/Loss', train_loss, epoch)
    writer_finetune.add_scalar('Train/Accuracy', train_acc, epoch)
    writer_finetune.add_scalar('Train/F1', train_f1, epoch)
    writer_finetune.add_scalar('Validation/Loss', val_loss, epoch)
    writer_finetune.add_scalar('Validation/Accuracy', val_acc, epoch)
    writer_finetune.add_scalar('Validation/F1', val_f1, epoch)
    writer_finetune.add_scalar('Validation/Precision', val_precision, epoch)
    writer_finetune.add_scalar('Validation/Recall', val_recall, epoch)

    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}')


Fine-tuning Epoch 1/5
--------------------------------------------------


Validation: 100%|██████████| 63/63 [02:51<00:00,  2.72s/it]


Train Loss: 0.1218, Acc: 0.9537, F1: 0.9537
Val Loss: 0.0714, Acc: 0.9725, F1: 0.9725, Precision: 0.9726, Recall: 0.9725

Fine-tuning Epoch 2/5
--------------------------------------------------


Validation: 100%|██████████| 63/63 [00:16<00:00,  3.75it/s]


Train Loss: 0.1001, Acc: 0.9597, F1: 0.9597
Val Loss: 0.0727, Acc: 0.9750, F1: 0.9750, Precision: 0.9751, Recall: 0.9750

Fine-tuning Epoch 3/5
--------------------------------------------------


...

Train Loss: 0.0926, Acc: 0.9612, F1: 0.9612
Val Loss: 0.0712, Acc: 0.9780, F1: 0.9780, Precision: 0.9780, Recall: 0.9780

Fine-tuning Epoch 4/5
--------------------------------------------------


Validation: 100%|██████████| 63/63 [00:16<00:00,  3.84it/s]


Train Loss: 0.0907, Acc: 0.9632, F1: 0.9632
Val Loss: 0.0721, Acc: 0.9750, F1: 0.9750, Precision: 0.9750, Recall: 0.9750

Fine-tuning Epoch 5/5
--------------------------------------------------


Validation: 100%|██████████| 63/63 [00:16<00:00,  3.72it/s]

Train Loss: 0.0745, Acc: 0.9722, F1: 0.9722
Val Loss: 0.0731, Acc: 0.9745, F1: 0.9745, Precision: 0.9745, Recall: 0.9745


In [ ]:
torch.save(model_finetuned.state_dict(), 'saved_models/model_resnet18_finetuned.pth')
writer_finetune.close()

In [ ]:
finetuned_test_metrics = evaluate_model(model_finetuned, test_loader_2, device, "Test_2")

print(f"\nРезультаты дообученной модели на Test_2:")
print(f"Accuracy: {finetuned_test_metrics['accuracy']:.4f}")
print(f"F1 Score: {finetuned_test_metrics['f1_score']:.4f}")
print(f"Precision: {finetuned_test_metrics['precision']:.4f}")
print(f"Recall: {finetuned_test_metrics['recall']:.4f}")

Test_2 Evaluation: 100%|██████████| 63/63 [00:18<00:00,  3.42it/s]


Результаты дообученной модели на Test_2:
Accuracy: 0.9745
F1 Score: 0.9745
Precision: 0.9745
Recall: 0.9745


In [ ]:
dvc_config = {
    'stages': {
        'train_base': {
            'cmd': 'python train_base.py',
            'deps': [
                'ai-vs-human-generated-dataset-hw/Train_1/train.csv',
                'ai-vs-human-generated-dataset-hw/Train_1'
            ],
            'outs': ['saved_models/model_resnet18_base.pth'],
            'metrics': ['logs/test1_metrics.json']
        },
        'upload_to_s3': {
            'cmd': 'python upload_to_s3.py --model base',
            'deps': ['saved_models/model_resnet18_base.pth'],
            'outs': []
        },
        'finetune': {
            'cmd': 'python finetune_model.py',
            'deps': [
                'ai-vs-human-generated-dataset-hw/Train_2/train.csv',
                'ai-vs-human-generated-dataset-hw/Train_2',
                'saved_models/model_resnet18_base.pth'
            ],
            'outs': ['saved_models/model_resnet18_finetuned.pth'],
            'metrics': ['logs/test2_metrics.json']
        },
        'upload_finetuned_to_s3': {
            'cmd': 'python upload_to_s3.py --model finetuned',
            'deps': ['saved_models/model_resnet18_finetuned.pth'],
            'outs': []
        },
        'compare_models': {
            'cmd': 'python compare_models.py',
            'deps': [
                'saved_models/model_resnet18_base.pth',
                'saved_models/model_resnet18_finetuned.pth'
            ],
            'metrics': ['logs/comparison_results.json']
        }
    }
}

In [ ]:
with open('dvc.yaml', 'w') as f:
    yaml.dump(dvc_config, f, default_flow_style=False)

In [ ]:
with open('train_base.py', 'w') as f:
    f.write("""
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script', 'train_model.ipynb'])
subprocess.run(['python', '-c', "exec(open('train_model.py').read())"])
""")

In [ ]:
with open('upload_to_s3.py', 'w') as f:
    f.write("""import sys
import boto3
from botocore.client import Config

model_type = sys.argv[1] if len(sys.argv) > 1 else 'base'
S3_CONFIG = {
    'endpoint': 'http://localhost:9000',
    'access_key': 'minioadmin',
    'secret_key': 'minioadmin',
    'bucket': 'ml-models'
}
s3_client = boto3.client('s3', endpoint_url=S3_CONFIG['endpoint'],
                         aws_access_key_id=S3_CONFIG['access_key'],
                         aws_secret_access_key=S3_CONFIG['secret_key'],
                         config=Config(signature_version='s3v4'), verify=False)
if model_type == 'base':
    s3_client.upload_file('saved_models/model_resnet18_base.pth', S3_CONFIG['bucket'], 'models/model_resnet18_base.pth')
else:
    s3_client.upload_file('saved_models/model_resnet18_finetuned.pth', S3_CONFIG['bucket'], 'models/model_resnet18_finetuned.pth')
print(f"{model_type} model uploaded to S3")
""")

In [ ]:
with open('finetune_model.py', 'w') as f:
    f.write("""
import subprocess
subprocess.run(['python', '-c', "exec(open('finetune_code.py').read())"])
""")

In [ ]:
with open('compare_models.py', 'w') as f:
    f.write("""import json
import numpy as np

with open('logs/test1_metrics.json', 'r') as f:
    base_metrics = json.load(f)
with open('logs/test2_metrics.json', 'r') as f:
    finetuned_metrics = json.load(f)

comparison = {
    'base_model': base_metrics,
    'finetuned_model': finetuned_metrics
}

with open('logs/comparison_results.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print("Comparison results saved to logs/comparison_results.json")
""")

## 10. Напишите вывод о полученных результатах

1. Базовая модель ResNet18 показало хорошее качество 97% и успешно справляется с задачей классификации
2. Дообучение на дополнительных данных позволило модели
     улучшить качество до 97.45%